In [ ]:
# Importing the necessary libraries

import pandas as pd

# Loading the dataset

In [ ]:
df = pd.read_csv('data.csv')

# Inspecting the dataset

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# Shape

df.shape

In [ ]:
# Coulumns of the dataset

df.columns

In [ ]:
# Few rows of the dataset

df.head()

In [ ]:
df.info()

In [ ]:
# Checking for missing value
df.isnull().sum()

In [ ]:
# Checking for duplicate rows
df.duplicated().sum()

# Exploring the Data

In [ ]:
label_count = df['label'].value_counts()
label_count


In [ ]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})
df

# Cleaning the text

In [ ]:
import nltk
import re

# nltk.download("stopwords")
# nltk.download("punkt")

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [ ]:
def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove punctuation and numbers
    text = re.sub(r"[^a-z\s]", "", text)

    # tokenize
    words = text.split()

    # remove stopwords
    words = [word for word in words if word not in stop_words]

    # stemming
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)



In [ ]:
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

In [ ]:
X = df["clean_text"]
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
nb_pred = nb.predict(X_test)

lr_pred = lr.predict(X_test)

rf_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
def evaluate(name, y_true, y_pred):

    print(f"\n{name}")

    print("Accuracy :", accuracy_score(y_true, y_pred))

    print("Precision:", precision_score(y_true, y_pred))

    print("Recall   :", recall_score(y_true, y_pred))

    print("F1 Score :", f1_score(y_true, y_pred))

    print("\nConfusion Matrix")

    print(confusion_matrix(y_true, y_pred))

In [ ]:
evaluate("Naive Bayes", y_test, nb_pred)

evaluate("Logistic Regression", y_test, lr_pred)

evaluate("Random Forest", y_test, rf_pred)

In [ ]:
results = []

models = {
    "Naive Bayes": nb,
    "Logistic Regression": lr,
    "Random Forest": rf
}

for name, model in models.items():

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]

results_df.set_index("Model")[metrics].plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0.6, 1.0)
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

models = {
    "Naive Bayes": nb,
    "Logistic Regression": lr,
    "Random Forest": rf
}

for name, model in models.items():

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test
    )

    plt.title(name)

    plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(7,7))

RocCurveDisplay.from_estimator(nb, X_test, y_test)

RocCurveDisplay.from_estimator(lr, X_test, y_test)

RocCurveDisplay.from_estimator(rf, X_test, y_test)

plt.title("ROC Curve Comparison")

plt.show()